<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_CRVG_Securitization_ACTP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# FRTB Advanced Standardised Approach: Vega Risk Calculation
# -----------------------------------------------------------------------------
# This script calculates the Vega Risk capital requirement for a portfolio of
# CSR Securitisations in the ACTP, following the FRTB framework.
# The calculations are broken down into cells that correspond to the steps
# outlined in the HTML report.

import pandas as pd
import numpy as np

# -----------------------------------------------------------------------------
# Cell 1: Setup and Initial Portfolio (Corresponds to HTML Steps 1 & 2)
# -----------------------------------------------------------------------------
# This cell defines the initial portfolio data and the key regulatory parameters
# required for the calculation.

# --- Initial Portfolio Data ---
portfolio_data = [
    {'position_id': 1, 'bucket': 3, 'sector': 'Financials', 'tenor_str': '1Y', 'issuer': 'Issuer 1', 'tranche': 'Tranche 1', 'gross_sensitivity': 35000},
    {'position_id': 2, 'bucket': 3, 'sector': 'Financials', 'tenor_str': '3Y', 'issuer': 'Issuer 2', 'tranche': 'Tranche 2', 'gross_sensitivity': -45000},
    {'position_id': 3, 'bucket': 5, 'sector': 'Consumer', 'tenor_str': '1Y', 'issuer': 'Issuer 3', 'tranche': 'Tranche 3', 'gross_sensitivity': 20000},
    {'position_id': 4, 'bucket': 5, 'sector': 'Consumer', 'tenor_str': '5Y', 'issuer': 'Issuer 4', 'tranche': 'Tranche 4', 'gross_sensitivity': 50000}
]

# --- Regulatory Parameters ---
TENOR_TO_YEARS = {'1Y': 1, '3Y': 3, '5Y': 5}
VEGA_RISK_WEIGHT = 1.00 # As per Article 325ax for CSR ACTP
# Delta correlation components for intra-bucket calculation (Article 325ai)
RHO_NAME_DIFF = 0.35
RHO_TENOR_DIFF = 0.65
RHO_BASIS_DIFF = 0.99
# Alpha for option maturity correlation (Article 325ay)
ALPHA = 0.01
# Cross-bucket correlation (Article 325aj, Table 5)
GAMMA_3_5 = 0.15

# Create the initial DataFrame
df_portfolio = pd.DataFrame(portfolio_data)
df_portfolio['tenor_years'] = df_portfolio['tenor_str'].map(TENOR_TO_YEARS)

print("--- Step 1 & 2: Initial Portfolio and Gross Sensitivities ---")
print("The calculation begins with the initial positions. Each position is a unique risk factor.")
print(df_portfolio[['position_id', 'bucket', 'sector', 'issuer', 'tenor_str', 'tranche', 'gross_sensitivity']].to_string(index=False))
print("-" * 70)


# -----------------------------------------------------------------------------
# Cell 2: Net Sensitivities (Corresponds to HTML Step 3)
# -----------------------------------------------------------------------------
# As each position has a unique combination of issuer, tenor, and tranche,
# no netting is possible. The net sensitivity equals the gross sensitivity.

df_net = df_portfolio.copy()
df_net.rename(columns={'gross_sensitivity': 'net_sensitivity'}, inplace=True)

print("\n--- Step 3: Net Sensitivities ---")
print("Since all risk factors are unique, no netting occurs.")
print(df_net[['bucket', 'issuer', 'tenor_str', 'tranche', 'net_sensitivity']].to_string(index=False))
print("-" * 70)


# -----------------------------------------------------------------------------
# Cell 3: Weighted Sensitivities (Corresponds to HTML Step 4)
# -----------------------------------------------------------------------------
# Apply the regulatory risk weight to the net sensitivities. We also calculate
# the sum of weighted sensitivities (Sb) for each bucket.

df_weighted = df_net.copy()
df_weighted['risk_weight'] = VEGA_RISK_WEIGHT
df_weighted['weighted_sensitivity'] = df_weighted['net_sensitivity'] * df_weighted['risk_weight']

# Calculate Sb for each bucket
s_b_values = df_weighted.groupby('bucket')['weighted_sensitivity'].sum().to_dict()
S3 = s_b_values.get(3, 0)
S5 = s_b_values.get(5, 0)

print("\n--- Step 4: Weighted Sensitivities ---")
print(f"Applying a {VEGA_RISK_WEIGHT:%} risk weight as per Article 325ax.")
print(df_weighted[['bucket', 'issuer', 'net_sensitivity', 'risk_weight', 'weighted_sensitivity']].to_string(index=False))
print(f"\nSum of Weighted Sensitivities (S3): {S3:,.2f}")
print(f"Sum of Weighted Sensitivities (S5): {S5:,.2f}")
print("-" * 70)


# -----------------------------------------------------------------------------
# Cell 4: Intra-Bucket Correlation (Corresponds to HTML Step 5)
# -----------------------------------------------------------------------------
# Determine the correlation coefficients for risk factors within the same bucket.

# Delta correlation component
rho_delta_comp = RHO_NAME_DIFF * RHO_TENOR_DIFF * RHO_BASIS_DIFF

# Option maturity component for Bucket 3 (1Y vs 3Y)
t1, t2 = 1, 3
rho_maturity_b3 = np.exp(-ALPHA * abs(t1 - t2) / min(t1, t2))
rho_kl_b3 = min(rho_delta_comp * rho_maturity_b3, 1)

# Option maturity component for Bucket 5 (1Y vs 5Y)
t3, t4 = 1, 5
rho_maturity_b5 = np.exp(-ALPHA * abs(t3 - t4) / min(t3, t4))
rho_kl_b5 = min(rho_delta_comp * rho_maturity_b5, 1)

print("\n--- Step 5: Intra-Bucket Correlation Determination ---")
print(f"Delta Component (name * tenor * basis): {rho_delta_comp:.4f}")
print(f"Maturity Component (Bucket 3): {rho_maturity_b3:.4f}")
print(f"Final Correlation for Bucket 3 (1Y vs 3Y): {rho_kl_b3:.2%}")
print(f"Maturity Component (Bucket 5): {rho_maturity_b5:.4f}")
print(f"Final Correlation for Bucket 5 (1Y vs 5Y): {rho_kl_b5:.2%}")
print("-" * 70)


# -----------------------------------------------------------------------------
# Cell 5: Cross-Bucket Correlation (Corresponds to HTML Step 6)
# -----------------------------------------------------------------------------
# State the correlation between Bucket 3 (Financials) and Bucket 5 (Consumer).

print("\n--- Step 6: Cross-Bucket Correlation Determination ---")
print(f"The correlation between Bucket 3 and Bucket 5 is {GAMMA_3_5:.2%} as per Article 325aj, Table 5.")
print("-" * 70)


# -----------------------------------------------------------------------------
# Cell 6: Intra-Bucket Aggregation (Corresponds to HTML Step 7)
# -----------------------------------------------------------------------------
# Calculate the bucket-specific capital charge (Kb) for the Medium Scenario.

def calculate_k_bucket(df_bucket, rho_kl):
    ws_values = df_bucket['weighted_sensitivity'].values
    sum_ws_sq = np.sum(ws_values**2)
    cross_term = 2 * rho_kl * ws_values[0] * ws_values[1]
    return np.sqrt(max(0, sum_ws_sq + cross_term))

K3 = calculate_k_bucket(df_weighted[df_weighted['bucket'] == 3], rho_kl_b3)
K5 = calculate_k_bucket(df_weighted[df_weighted['bucket'] == 5], rho_kl_b5)

print("\n--- Step 7: Intra-Bucket Aggregation (Medium Scenario) ---")
print(f"Bucket 3 Capital (K3): {K3:,.2f}")
print(f"Bucket 5 Capital (K5): {K5:,.2f}")
print("-" * 70)


# -----------------------------------------------------------------------------
# Cell 7: Cross-Bucket Aggregation (Corresponds to HTML Step 8)
# -----------------------------------------------------------------------------
# Aggregate bucket capital to get the total for the Medium Scenario.

sum_k_sq = K3**2 + K5**2
cross_bucket_term = 2 * GAMMA_3_5 * S3 * S5
capital_medium = np.sqrt(max(0, sum_k_sq + cross_bucket_term))

print("\n--- Step 8: Cross-Bucket Aggregation (Medium Scenario) ---")
print(f"Sum of Squared K-buckets: {sum_k_sq:,.2f}")
print(f"Cross-Bucket Term: {cross_bucket_term:,.2f}")
print(f"Total Capital (Medium Scenario): {capital_medium:,.2f}")
print("-" * 70)


# -----------------------------------------------------------------------------
# Cell 8: Correlation Scenarios (Corresponds to HTML Step 9)
# -----------------------------------------------------------------------------
# Recalculate the total capital for High and Low correlation scenarios.

def calculate_total_capital(rho_b3, rho_b5, gamma_3_5):
    """Helper function to recalculate capital for a given set of correlations."""
    k3_scen = calculate_k_bucket(df_weighted[df_weighted['bucket'] == 3], rho_b3)
    k5_scen = calculate_k_bucket(df_weighted[df_weighted['bucket'] == 5], rho_b5)
    sum_k_sq_scen = k3_scen**2 + k5_scen**2
    cross_term_scen = 2 * gamma_3_5 * S3 * S5
    return np.sqrt(max(0, sum_k_sq_scen + cross_term_scen))

# High Correlation Scenario
rho_kl_b3_high = min(rho_kl_b3 * 1.25, 1)
rho_kl_b5_high = min(rho_kl_b5 * 1.25, 1)
gamma_3_5_high = min(GAMMA_3_5 * 1.25, 1)
capital_high = calculate_total_capital(rho_kl_b3_high, rho_kl_b5_high, gamma_3_5_high)

# Low Correlation Scenario
rho_kl_b3_low = max(2 * rho_kl_b3 - 1, 0.75 * rho_kl_b3)
rho_kl_b5_low = max(2 * rho_kl_b5 - 1, 0.75 * rho_kl_b5)
gamma_3_5_low = max(2 * GAMMA_3_5 - 1, 0.75 * GAMMA_3_5)
capital_low = calculate_total_capital(rho_kl_b3_low, rho_kl_b5_low, gamma_3_5_low)

scenario_results = pd.DataFrame([
    {'Scenario': 'Medium', 'Capital Requirement': capital_medium},
    {'Scenario': 'High', 'Capital Requirement': capital_high},
    {'Scenario': 'Low', 'Capital Requirement': capital_low}
])

print("\n--- Step 9: Correlation Scenario Analysis ---")
print(scenario_results.to_string(index=False))
print("-" * 70)


# -----------------------------------------------------------------------------
# Cell 9: Final Charge Calculation (Corresponds to HTML Step 10)
# -----------------------------------------------------------------------------
# The final charge is the maximum of the three scenarios.

final_charge = scenario_results['Capital Requirement'].max()
winning_scenario = scenario_results.loc[scenario_results['Capital Requirement'].idxmax()]['Scenario']

print("\n--- Step 10: Final Capital Charge Calculation ---")
print(f"The final requirement is the maximum of the three scenarios.")
print("\n-------------------------------------------------")
print(f" Final Vega Capital Requirement: {final_charge:,.2f}")
print(f" (Driven by the {winning_scenario} Correlation Scenario)")
print("-------------------------------------------------")

--- Step 1 & 2: Initial Portfolio and Gross Sensitivities ---
The calculation begins with the initial positions. Each position is a unique risk factor.
 position_id  bucket     sector   issuer tenor_str   tranche  gross_sensitivity
           1       3 Financials Issuer 1        1Y Tranche 1              35000
           2       3 Financials Issuer 2        3Y Tranche 2             -45000
           3       5   Consumer Issuer 3        1Y Tranche 3              20000
           4       5   Consumer Issuer 4        5Y Tranche 4              50000
----------------------------------------------------------------------

--- Step 3: Net Sensitivities ---
Since all risk factors are unique, no netting occurs.
 bucket   issuer tenor_str   tranche  net_sensitivity
      3 Issuer 1        1Y Tranche 1            35000
      3 Issuer 2        3Y Tranche 2           -45000
      5 Issuer 3        1Y Tranche 3            20000
      5 Issuer 4        5Y Tranche 4            50000
------------------